In [ ]:
import random
import time
import math

import pandas as pd
from pandas import DataFrame
import numpy as np
from scipy.spatial import cKDTree


from preferences import prefs
from user import user_prefs

# startPoint = {"lat": 30.317504,"lon": 59.927085}
# endPoint = {"lat": 30.327108, "lon": 59.935408}

POPULATION_SIZE = 25
GENERATIONS = 50
MUTATION_RATE = 0.25

MIN_ROUTE_POINTS = 3
MAX_ROUTE_POINTS = 25

KILLOMETER_RADIUS = 1
ADD_POINT_KILLOMETER_RADIUS = 1

PRINT = False

INTEREST_K = 10
DISTANCE_K = 0.1

random.seed(42)
np.random.seed(42)

# Особь:
# {
#     "route": [индексы точек из df],
#     "fitness": число
# }


In [47]:
# df = pd.read_csv('data/places.csv')

# coords = df[["lat", "lon"]].to_numpy()
# tree = cKDTree(coords)

**Алгоритмы генетики**

In [48]:
from haversine import haversine

# Оптимизация конкретного маршрута
def optimize_route(route_ids, df: DataFrame, startPoint):
    points = (
        df[df["id"].isin(route_ids)]
        [["id", "lat", "lon"]]
        .copy()
    )

    remaining = points.to_dict("records")

    current = {
        "lat": startPoint["lat"],
        "lon": startPoint["lon"]
    }

    optimized_route = []

    while remaining:
        nearest = min(
            remaining,
            key=lambda p: haversine(
                (current["lat"], current["lon"]),
                (p["lat"], p["lon"])
            )
        )

        optimized_route.append(nearest["id"])

        current = nearest

        remaining.remove(nearest)

    return optimized_route
    
df_filtered=0
# Генерация популяции
def create_population(df: DataFrame, startPoint: dict[str, float], endPoint: dict[str, float], num: int):
    global df_filtered
    route_size = random.randint(
        MIN_ROUTE_POINTS,
        MAX_ROUTE_POINTS
    )

    radius_deg = KILLOMETER_RADIUS / 111.0

    start = np.array([
        startPoint["lat"],
        startPoint["lon"]
    ])

    finish = np.array([
        endPoint["lat"],
        endPoint["lon"]
    ])

    # Длина маршрута в километрах
    mean_lat = (start[0] + finish[0]) / 2

    lat_km = (finish[0] - start[0]) * 111
    lon_km = (
        (finish[1] - start[1])
        * 111
        * np.cos(np.radians(mean_lat))
    )

    route_length_km = np.hypot(lat_km, lon_km)

    # Точка каждые ~500 метров
    samples_count = max(
        2,
        int(route_length_km / 0.5)
    )

    route_points = np.linspace(
        start,
        finish,
        samples_count
    )

    indices = set()

    for point in route_points:
        nearby = tree.query_ball_point(
            point,
            r=radius_deg
        )

        indices.update(nearby)

    df_filtered = df.iloc[list(indices)]

    df_indexes = df_filtered["id"].tolist()

    if len(df_indexes) < route_size:
        route_size = len(df_indexes)

    population = []

    for _ in range(num):
        route_ids = random.sample(
            df_indexes,
            route_size
        )

        new_route = optimize_route(
            route_ids,
            df,
            startPoint
        )

        population.append({
            "route": new_route,
            "fitness": None
        })

    return population

# Длина маршрута
def route_distance(df: DataFrame, route: list, startPoint: dict[str, float], endPoint: dict[str, float]):
    
    route_df = pd.DataFrame()

    for j in route:
        route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

    remaining = route_df.to_dict(orient='index')
    current = {
        "lat": startPoint["lat"],
        "lon": startPoint["lon"]
    }

    distance = haversine((current["lat"], current["lon"]),
                         (remaining[0]["lat"], remaining[0]["lon"]))

    current = remaining[0]

    for point_id in remaining.keys():
        if point_id == 0:
            continue
        distance += haversine((current["lat"], current["lon"]),
                              (remaining[point_id]["lat"], remaining[point_id]["lon"]))
        current = remaining[point_id]
    distance += haversine((current["lat"], current["lon"]),
                         (endPoint["lat"], endPoint["lon"]))
    return distance


# Интересность конкретной точки
def point_interest_score(point: dict, unique_user_prefs: dict[str, int]):
    point_tags = next(iter(point.values()))
    score = 0
    for category, tags in prefs.items():
        weight = unique_user_prefs[category]
        for tag in tags:
            if tag in point_tags and point_tags[tag]:
                score += weight
    return score

# Фитнес-функция
def fitness_function(df: DataFrame, route: list, unique_user_prefs: dict[str, int], startPoint: dict[str, float], endPoint: dict[str, float]):

    total_interest = 0

    for place_id in route:
        point = df.iloc[(df["id"] == place_id)].to_dict(orient='index')
        total_interest += point_interest_score(point, unique_user_prefs)

    distance = route_distance(df, route, startPoint, endPoint)

    fitness = total_interest * INTEREST_K - distance * DISTANCE_K

    return fitness

# Проведение оценки популяции (просчёт фитнес-функций каждой особи)
def evaluate_population(df: DataFrame, population: dict, unique_user_prefs: dict[str, int], startPoint: dict[str, float], endPoint: dict[str, float]):
    for individual in population:
        individual["fitness"] = fitness_function(
            df,
            individual["route"],
            unique_user_prefs,
            startPoint,
            endPoint
        )


# Мутация
def mutate(df: DataFrame, individual, unique_user_prefs: dict[str, int], startPoint: dict[str, float]):
    
    if random.random() > MUTATION_RATE:
        return individual
    
    route = individual["route"][:]
    route_df = pd.DataFrame()

    for j in route:
        route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

    mutation_type = random.choice([
        "remove_minimum_score",
        "remove_most_distant_path",
        "add_nearby_point"
    ])
    # mutation_type = "add_nearby_point"
    if len(route) >= MAX_ROUTE_POINTS:
        mutation_type = random.choice([
            "remove_minimum_score",
            "remove_most_distant_path"
        ])
    
    if mutation_type == "remove_minimum_score":
        if len(route) > MIN_ROUTE_POINTS:
            min_score_point = route[0]
            
            min_score = point_interest_score({0:df.iloc[0].to_dict()}, unique_user_prefs)
            for place_id in route:
                new_min_score = point_interest_score({0:df.iloc[(df["id"] == place_id)].to_dict()}, unique_user_prefs)
                if min_score > new_min_score:
                    min_score = new_min_score
                    min_score_point = place_id
            
            route.remove(min_score_point)
    
    elif mutation_type == "remove_most_distant_path":
        if len(route) > MIN_ROUTE_POINTS:
            
            current = {"lat": startPoint["lat"], "lon": startPoint["lon"]}
            
            max_distance_path_point = route[0]
            max_distance_path = haversine((current["lat"], current["lon"]), (route_df["lat"].iloc[0], route_df["lon"].iloc[0])) +\
                                haversine((route_df["lat"].iloc[0], route_df["lon"].iloc[0]), (route_df["lat"].iloc[1], route_df["lon"].iloc[1]))
            
            
            for place_id in range(1, len(route)-1):
                place = route_df.loc[route_df["id"] == route[place_id]].iloc[0]
                second_place = route_df.loc[route_df["id"] == route[place_id+1]].iloc[0]
                new_max_distance_path = haversine((current["lat"], current["lon"]),(place["lat"], place["lon"])) +\
                                        haversine((place["lat"], place["lon"]),(second_place["lat"], second_place["lon"]))
                if max_distance_path < new_max_distance_path:
                    max_distance_path = new_max_distance_path
                    max_distance_path_point = route[place_id]
                current = {"lat": place["lat"], "lon": place["lon"]}
            
            route.remove(max_distance_path_point)
    
    elif mutation_type == "add_nearby_point":
        if len(route) < MAX_ROUTE_POINTS:
            source_id = random.choice(route)

            source_point = route_df.loc[
                route_df["id"] == source_id
            ].iloc[0]

            lat = source_point["lat"]
            lon = source_point["lon"]

            radius_deg = ADD_POINT_KILLOMETER_RADIUS / 111.0

            indices = tree.query_ball_point(
                [lat, lon],
                r=radius_deg
            )

            neighbors_df = df.iloc[indices]

            # id найденных точек
            neighbors_id = neighbors_df["id"].tolist()

            # исключаем уже существующие в маршруте
            candidates = [
                place_id
                for place_id in neighbors_id
                if place_id not in route
            ]

            if candidates:
                best_point = max(candidates,
                                key=lambda place_id:
                                    point_interest_score(
                                        {0: df.loc[df["id"] == place_id].iloc[0].to_dict()},
                                        unique_user_prefs
                                    )
                                )
                route.append(best_point)
    
    route = optimize_route(route, df, startPoint)
    
    return {"route": route, "fitness": None}

# Создание нового поколения
def create_next_generation(df: DataFrame, population: dict, unique_user_prefs: dict[str, int], startPoint: dict[str, float]):
    new_population = []

    population.sort(
        key=lambda x: x["fitness"],
        reverse=True
    )

    # elite = population[:math.ceil(POPULATION_SIZE/10)]
    elite = population[:10]

    new_population.extend(elite[:3])

    while len(new_population) < POPULATION_SIZE:
        child = mutate(df, random.choice(elite), unique_user_prefs, startPoint)
        new_population.append(child)

    return new_population

In [49]:
import folium

def hex_to_rgb(hex_str):
    """Преобразует HEX в кортеж RGB (0-255)"""
    hex_str = hex_str.lstrip('#')
    return tuple(int(hex_str[i:i+2], 16) for i in (0, 2, 4))

def rgb_to_hex(rgb):
    """Преобразует RGB в HEX строку"""
    return '#{:02x}{:02x}{:02x}'.format(int(rgb[0]), int(rgb[1]), int(rgb[2]))

def generate_gradient(start_hex, end_hex, steps):
    """Генерирует список цветов для градиента"""
    start_rgb = hex_to_rgb(start_hex)
    end_rgb = hex_to_rgb(end_hex)
    
    gradient_colors = []
    for i in range(steps):
        t = i / max(1, steps - 1)
        r = start_rgb[0] + (end_rgb[0] - start_rgb[0]) * t
        g = start_rgb[1] + (end_rgb[1] - start_rgb[1]) * t
        b = start_rgb[2] + (end_rgb[2] - start_rgb[2]) * t
        gradient_colors.append(rgb_to_hex((r, g, b)))
        
    return gradient_colors

def visual(df: DataFrame, all_routes: bool, population: dict):

    path_colors = generate_gradient("#FF0000","#0000FF", len(population))

    route_df = df[(df["id"].isin(population[0]["route"]))]
    m = folium.Map(
        location=[
            route_df["lat"].mean(),
            route_df["lon"].mean()
        ],
        zoom_start=14
    )
    if all_routes:
        for i in range(len(population)):
            individual = population[i]
            
            route_df = pd.DataFrame()

            for j in individual["route"]:
                route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

            # Точки маршрута
            folium.Marker(
                [startPoint["lat"], startPoint["lon"]]
            ).add_to(m)
            folium.Marker(
                [endPoint["lat"], endPoint["lon"]]
            ).add_to(m)

            route_line = [[startPoint["lat"], startPoint["lon"]]]
            route_line.extend(route_df[["lat", "lon"]].values.tolist())
            route_line.append([endPoint["lat"], endPoint["lon"]])

            # Линия маршрута
            folium.PolyLine(
                route_line,
                weight=4,
                color=path_colors[i]
            ).add_to(m)

        m.save(r"visualisations\route.html")
    else:
        individual = population[0]

        route_df = pd.DataFrame()

        for j in individual["route"]:
            route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

        # Точки маршрута
        folium.Marker(
            [startPoint["lat"], startPoint["lon"]]
        ).add_to(m)
        folium.Marker(
            [endPoint["lat"], endPoint["lon"]]
        ).add_to(m)

        route_line = [[startPoint["lat"], startPoint["lon"]]]
        route_line.extend(route_df[["lat", "lon"]].values.tolist())
        route_line.append([endPoint["lat"], endPoint["lon"]])

        # Линия маршрута
        folium.PolyLine(
            route_line,
            weight=4,
            color=path_colors[0]
        ).add_to(m)

        m.save(r"visualisations\route.html")

def visual_df_filtered(startPoint):

    m = folium.Map(
        location=[
            startPoint["lat"],
            startPoint["lon"]
        ],
        zoom_start=15
    )
    
    # Точки маршрута
    for i in df_filtered.values:
        folium.Marker(
            [i[1], i[2]]
        ).add_to(m)

    m.save(r"visualisations\filtered_points.html")

In [50]:
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

def render_generation_fast(
    df,
    generation_idx,
    population,
    startPoint,
    endPoint,
    all_routes=True
):
    output_dir = Path("visualisations/jpg")
    output_dir.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(10, 10))


    used_points = []

    def draw_route(individual, color):
        route_ids = individual["route"]
        route_df = df[df["id"].isin(route_ids)].copy()

        # порядок маршрута (быстро через dict вместо index)
        order_map = {id_: i for i, id_ in enumerate(route_ids)}
        route_df["order"] = route_df["id"].map(order_map)
        route_df = route_df.sort_values("order")

        xs = [startPoint["lon"]] + route_df["lon"].tolist() + [endPoint["lon"]]
        ys = [startPoint["lat"]] + route_df["lat"].tolist() + [endPoint["lat"]]

        plt.plot(xs, ys, color=color, linewidth=2, alpha=0.9)
        plt.scatter(route_df["lon"], route_df["lat"], s=12, color=color, alpha=0.7)

        used_points.extend(route_df[["lon", "lat"]].values.tolist())

    colors = generate_gradient("#FF0000", "#0000FF", len(population[:5]))
    if all_routes:
        for i, individual in enumerate(population[:5]):
            draw_route(individual, colors[i])
        draw_route(population[0], colors[0])
    else:
        draw_route(population[0], colors[0])

    # старт / финиш
    plt.scatter(startPoint["lon"], startPoint["lat"], c="green", s=80)
    plt.scatter(endPoint["lon"], endPoint["lat"], c="red", s=80)

    all_lons = [p[0] for p in used_points] + [startPoint["lon"], endPoint["lon"]]
    all_lats = [p[1] for p in used_points] + [startPoint["lat"], endPoint["lat"]]

    if all_lons and all_lats:
        min_lon, max_lon = min(all_lons), max(all_lons)
        min_lat, max_lat = min(all_lats), max(all_lats)

        padding_lon = (max_lon - min_lon) * 0.2
        padding_lat = (max_lat - min_lat) * 0.2

        plt.xlim(min_lon - padding_lon, max_lon + padding_lon)
        plt.ylim(min_lat - padding_lat, max_lat + padding_lat)

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    plt.title(f"Generation {generation_idx} | Datetime: {now}")
    plt.tight_layout()

    path = output_dir / f"generation_{generation_idx}.jpg"
    plt.savefig(path, dpi=200)
    plt.close()

    return path

**Жадный алгоритм**

In [ ]:
# startPoint = {"lat": 59.927085,"lon": 30.317504}
# endPoint = {"lat": 59.948936, "lon": 30.374199}

# df = pd.read_csv('data/places.csv')
# coords = df[["lat", "lon"]].to_numpy()
# tree = cKDTree(coords)

# population = create_population(df, startPoint, endPoint, POPULATION_SIZE)

# df = df_filtered
# coords = df[["lat", "lon"]].to_numpy()
# tree = cKDTree(coords)

# def greedy_route(df, unique_user_prefs, route_size=15):
#     scores = []

#     for _, row in df.iterrows():
#         point_dict = row.to_dict()
        
#         score = point_interest_score(
#             {0:point_dict},
#             unique_user_prefs
#         )

#         scores.append(score)

#     df_scored = df.copy()
#     df_scored["interest_score"] = scores

#     df_sorted = df_scored.sort_values(
#         by="interest_score",
#         ascending=False
#     )

#     top_points = df_sorted.head(route_size)

#     route = top_points["id"].tolist()

#     return {
#         "route": route,
#         "fitness": None
#     }

# unique_user_prefs = {
#     "military": 5,
#     "religion": 1,
#     "architecture": 1,
#     "transport": 5,
#     "sight": 1,
#     "interactive": 2,
#     "nutrition": 1,
#     "housing": 1
#     }

# print("Запуск жадного алгоритма")

# start_time = time.time()
# individual = greedy_route(
#     df_filtered,
#     unique_user_prefs,
#     route_size=15
# )
# work_time = time.time() - start_time

# print("Время:", work_time)
# individual['fitness'] = fitness_function(
#             df_filtered,
#             individual["route"],
#             unique_user_prefs,
#             startPoint,
#             endPoint
#         )
# print(f"Лучшая фитнес-функция: {individual['fitness']:.10f}")
# print(f"Количество точек: {len(individual['route'])}")
# print(f"Длина маршрута: {route_distance(df_filtered, individual['route'], startPoint, endPoint)}")

# populat = [individual]
# unique_user_prefs = {
#     "military": 1,
#     "religion": 5,
#     "architecture": 5,
#     "transport": 1,
#     "sight": 5,
#     "interactive": 1,
#     "nutrition": 1,
#     "housing": 0
#     }

# print("Запуск жадного алгоритма")

# start_time = time.time()
# individual = greedy_route(
#     df_filtered,
#     unique_user_prefs,
#     route_size=15
# )
# work_time = time.time() - start_time

# print("Время:", work_time)
# individual['fitness'] = fitness_function(
#             df_filtered,
#             individual["route"],
#             unique_user_prefs,
#             startPoint,
#             endPoint
#         )
# print(f"Лучшая фитнес-функция: {individual['fitness']:.10f}")
# print(f"Количество точек: {len(individual['route'])}")
# print(f"Длина маршрута: {route_distance(df_filtered, individual['route'], startPoint, endPoint)}")
# populat.append(individual)

# print(populat)

# render_generation_fast(
#         df_filtered,
#         -1,
#         populat,
#         startPoint,
#         endPoint,
#         all_routes=True
#     )

Запуск жадного алгоритма
Время: 0.045211076736450195
Лучшая фитнес-функция: 148.3786148814
Количество точек: 15
Длина маршрута: 16.21385118584986
Запуск жадного алгоритма
Время: 0.046724557876586914
Лучшая фитнес-функция: 748.3786148814
Количество точек: 15
Длина маршрута: 16.21385118584986
[{'route': [5652382524, 12128370036, 9977900680, 1041154862, 5256619603, 11966324352, 5652382613, 5047040826, 2413998907, 2413998986, 6689900914, 10862756262, 3194321869, 4189483389, 4191691299], 'fitness': 148.37861488141502}, {'route': [5652382524, 12128370036, 9977900680, 1041154862, 5256619603, 11966324352, 5652382613, 5047040826, 2413998907, 2413998986, 6689900914, 10862756262, 3194321869, 4189483389, 4191691299], 'fitness': 748.378614881415}]


WindowsPath('visualisations/jpg/generation_-1.jpg')

**Запуск генетики**

In [52]:
# startPoint = {"lat": 59.927085,"lon": 30.317504}
# # endPoint = {"lat": 59.935408, "lon": 30.327108} # Станция метро на невском проспекте
# endPoint = {"lat": 59.948936, "lon": 30.374199}

# df = pd.read_csv('data/places.csv')
# coords = df[["lat", "lon"]].to_numpy()
# tree = cKDTree(coords)

# unique_user_prefs = user_prefs

# print("Запуск стандартного генетического алгоритма")

# print("Размер популяции:", POPULATION_SIZE)
# print("Количество поколений:", GENERATIONS)
# print("Шанс мутации:", MUTATION_RATE)

# # Создание популяции
# population = create_population(df, startPoint, endPoint, POPULATION_SIZE)
# df = df_filtered
# coords = df[["lat", "lon"]].to_numpy()
# tree = cKDTree(coords)

# # Визуализация отфильтрованных ячеек
# visual_df_filtered(startPoint)

# start_time = time.time()
# for generation in range(GENERATIONS):
#     evaluate_population(df_filtered, population, unique_user_prefs, startPoint, endPoint)

#     best = max(
#         population,
#         key=lambda x: x["fitness"]
#     )
#     print(
#         f"Поколение {generation} | "
#         f"Лучшая фитнес-функция: {best['fitness']:.10f} | "
#         f"Длина маршрута: {len(best['route'])}"
#     )
#     visual(df_filtered, True, population)
#     render_generation_fast(
#         df_filtered,
#         generation,
#         population,
#         startPoint,
#         endPoint,
#         all_routes=True
#     )
    
#     population = create_next_generation(df_filtered, population, unique_user_prefs, startPoint)

# work_time = time.time() - start_time
# print("Время:", work_time)

# evaluate_population(df_filtered, population, unique_user_prefs, startPoint, endPoint)
# best = max(
#         population,
#         key=lambda x: x["fitness"]
#     )
# print(f"Лучшая фитнес-функция: {best['fitness']:.10f}")
# print(f"Количество точек: {len(best['route'])}")
# print(f"Длина маршрута: {route_distance(df_filtered, best['route'], startPoint, endPoint)}")

# visual(df_filtered, False, population)
# render_generation_fast(
#         df_filtered,
#         -1,
#         population,
#         startPoint,
#         endPoint,
#         all_routes=False
#     )

**"ЭКСПЕРИМЕНТЫ**

In [ ]:
import time
import itertools
import pandas as pd


def run_experiments(
    df_filtered,
    startPoint,
    endPoint,
    unique_user_prefs,
    save_path=f"Сравнения\ga_experiments.csv"
):
    population_sizes = [25, 50, 100]
    generations_list = [25, 50, 100]
    mutation_rates = [0.1, 0.25, 0.5]
    # population_sizes = [25]
    # generations_list = [25]
    # mutation_rates = [0.1, 0.25, 0.5]

    results = []

    experiment_id = 0

    for POPULATION_SIZE, GENERATIONS, MUTATION_RATE in itertools.product(
        population_sizes,
        generations_list,
        mutation_rates
    ):
        experiment_id += 1

        print(f"\n=== EXPERIMENT {experiment_id} ===")
        print(f"POPULATION_SIZE={POPULATION_SIZE}, GENERATIONS={GENERATIONS}, MUTATION_RATE={MUTATION_RATE}")

        population = create_population(
            df_filtered,
            startPoint,
            endPoint,
            POPULATION_SIZE
        )

        coords = df_filtered[["lat", "lon"]].to_numpy()
        tree = cKDTree(coords)

        start_time = time.time()

        best_fitness = None
        best_route_len = None
        best_distance = None

        for generation in range(GENERATIONS):

            evaluate_population(
                df_filtered,
                population,
                unique_user_prefs,
                startPoint,
                endPoint
            )

            best = max(population, key=lambda x: x["fitness"])

            if best_fitness is None or best["fitness"] > best_fitness:
                best_fitness = best["fitness"]
                best_route_len = len(best["route"])
                best_distance = route_distance(
                    df_filtered,
                    best["route"],
                    startPoint,
                    endPoint
                )

            population = create_next_generation(
                df_filtered,
                population,
                unique_user_prefs,
                startPoint
            )

        work_time = time.time() - start_time

        # финальная переоценка
        evaluate_population(
            df_filtered,
            population,
            unique_user_prefs,
            startPoint,
            endPoint
        )

        best = max(population, key=lambda x: x["fitness"])

        final_distance = route_distance(
            df_filtered,
            best["route"],
            startPoint,
            endPoint
        )

        results.append({
            "experiment_id": experiment_id,
            "population_size": POPULATION_SIZE,
            "generations": GENERATIONS,
            "mutation_rate": MUTATION_RATE,
            "best_fitness": best_fitness,
            "final_fitness": best["fitness"],
            "best_route_length": best_route_len,
            "final_route_length": len(best["route"]),
            "best_distance": best_distance,
            "final_distance": final_distance,
            "time_sec": work_time
        })

        print("TIME:", work_time)
        print("BEST FITNESS:", best["fitness"])
        print("BEST DISTANCE:", final_distance)

    # --- save results ---
    results_df = pd.DataFrame(results)
    results_df.to_csv(save_path, index=False)

    print(f"\nSaved results to {save_path}")

    return results_df

In [ ]:
# 1. Фильтрация данных
df = df_filtered

# 2. Построение KDTree (если используется дальше внутри GA)
coords = df[["lat", "lon"]].to_numpy()
tree = cKDTree(coords)

results_df = run_experiments(
    df_filtered=df_filtered,
    startPoint=startPoint,
    endPoint=endPoint,
    unique_user_prefs=unique_user_prefs,
    save_path=f"Сравнения\ga_experiments.csv"
)